# Raw archive inspection

Thin notebook over `sarb.data.inspect_zip` for USA and crypto raw zip archives.

In [ ]:
from pathlib import Path

from sarb.config import PROJECT_ROOT
from sarb.data.inspect_zip import format_report, inspect_archives

archives = [
    PROJECT_ROOT / "data/raw/usa_data.zip",
    PROJECT_ROOT / "data/raw/crypto_data.zip",
]
reports = inspect_archives(archives)
for report in reports:
    print(format_report(report))
    print()

In [ ]:
import pandas as pd

summary_rows = []
for report in reports:
    for asset in report.assets:
        summary_rows.append(
            {
                "archive": Path(report.archive).name,
                "symbol": asset.symbol,
                "rows": asset.row_count,
                "missing": sum(asset.missing_by_column.values()),
                "first_timestamp": asset.first_timestamp,
                "last_timestamp": asset.last_timestamp,
                "primary_frequency": asset.primary_frequency,
                "frequency_mode_share": asset.frequency_mode_share,
                "irregular_gaps": asset.non_primary_gap_count,
            }
        )

summary = pd.DataFrame(summary_rows)
summary.groupby("archive").agg(
    assets=("symbol", "count"),
    min_rows=("rows", "min"),
    max_rows=("rows", "max"),
    total_missing=("missing", "sum"),
    earliest_start=("first_timestamp", "min"),
    latest_end=("last_timestamp", "max"),
)